# Motility assay

**What it does.** Track objects through a timelapse and summarise how far and how fast they moved.

**When to use it.** For timelapse acquisitions where the phenotype is movement.

**What you get.** Per-track trajectories, speed and displacement summaries.

---

> Every path below is a placeholder. Point `src` at your own data before running.
> Nothing in this notebook writes outside the folder you give it.

## 1. Check the install

If this cell fails, the rest cannot work. It reports the version and whether a GPU is visible — segmentation and training are usable on CPU but slow.

In [ ]:
import spacr
from spacr.version import version_str

print(version_str)

## 2. The function this notebook runs

`spacr.timelapse.automated_motility_assay`

```
automated_motility_assay(settings)
```

End-to-end merged-npy pipeline for cell/pathogen motility and infection QC.

In [ ]:
from spacr.timelapse import automated_motility_assay

## 3. Settings

`spacr.settings.get_automated_motility_assay_default_settings` fills in every default, so you only have to write down what differs. The cell below prints the full set as it exists in this version — treat that output as the reference, not this notebook.

Change values in `settings`, not in the defaults helper.

In [ ]:
from spacr.settings import get_automated_motility_assay_default_settings

defaults = get_automated_motility_assay_default_settings({})
for key in sorted(defaults):
    print(f'{key:38s} {defaults[key]!r}')

### Every setting this function accepts

Every key, with its default and what it controls. Edit values in place; delete nothing — a key left at its default behaves exactly as if it were absent.

Generated from this installed version by `tools/build_notebook_settings.py`, so these are the real keys, the real defaults and the real descriptions. Re-run that tool after upgrading spaCR.

In [ ]:
# Generated by tools/build_notebook_settings.py — do not edit by hand.
# Values are this installed version's defaults; the text above each key is
# its live description. Edit values in place -- a key left at its default
# behaves exactly as if it were absent.

# spacr.timelapse.automated_motility_assay  (55 settings)
settings = {
    # (int or None) - Zero-indexed raw acquisition channel that Cellpose
    # segments into cell masks; it also selects which channel the
    # cell_background, cell_Signal_to_noise and remove_background_cell
    # settings are applied to during preprocessing. Set to None and no
    # cell masks, cell table or cell crops are produced. At least one of
    # cell/nucleus/pathogen/organelle_channel must be an integer or the
    # run aborts. Default None.
    'cell_channel': 2,

    # (list of int) - Zero-indexed image channels kept in merged/*.npy and
    # measured by measure_crop; each entry produces its own
    # <object>_channel_<n>_* intensity columns. The list length fixes
    # where masks land, so cell/nucleus/pathogen_mask_dim must shift if
    # you change it. Preprocessing silently resets it to range(n) when it
    # does not match the number of channel folders found. Default
    # [0,1,2,3].
    'channels': [0, 1, 2, 3],

    # (str) - Table inside <src>/measurements/measurements.db that holds
    # the pre-QC per-frame measurements. It is rewritten with
    # if_exists='replace' on every run, a companion table with the suffix
    # '_well_motility' holds the well summary, and the same name is read
    # back when reuse_existing_measurements is True. Default
    # 'timelapse_object_measurements'.
    'db_table_name': 'timelapse_object_measurements',

    # (str) - What happens when the QC call disagrees with the mask-based
    # label: 'relabel' overwrites the label and keeps the cell, 'remove'
    # deletes the disagreeing cells outright. Use 'relabel' to preserve
    # sample size, 'remove' when you only want cells whose mask and
    # intensity evidence agree. Unknown values fall back to 'relabel'.
    # Default 'relabel'.
    'infection_intensity_mode': 'relabel',

    # (int) - Bin count for the pathogen-intensity histogram, clamped to
    # 10-256. The histogram strategy walks bins from low to high and takes
    # the first whose infected fraction reaches the target as its
    # intensity threshold, so more bins give a finer threshold but noisier
    # per-bin fractions. Also sets the QC panel histogram. Default 64.
    'infection_intensity_n_bins': 64,

    # (bool) - Save the infection-intensity histogram PNG and reserve the
    # QC sub-axes (histogram, embedding, or XGBoost probability plus
    # feature importance) inside the combined intensity/motility panel.
    # Set False to skip that plotting work on large runs; the infection
    # relabelling itself is unchanged either way. Default True.
    'infection_intensity_qc_graphs': True,

    # (str) - Whether infection QC is fitted once or per group:
    # 'combined'/'global'/'all' fits one model on everything,
    # 'plate'/'per_plate' one per plateID, 'well'/'per_well' one per
    # plate-well, and 'none'/'off' skips QC; an unrecognised string falls
    # back to combined behaviour with a warning. Per-well fitting absorbs
    # staining and exposure differences but needs enough cells per well;
    # every group still writes its own QC plot, only the QC payload
    # embedded in the summary panel is taken from the first processed
    # group. Default 'per_well'.
    'infection_intensity_qc_scope': 'per_well',

    # (str) - How infected vs uninfected is decided once
    # infection_intensity_qc is True: 'xgboost' trains a classifier on
    # intensity extremes, 'histogram' picks one intensity threshold, and
    # 'pca'/'umap'/'tsne' cluster a 2D embedding. Unknown values fall back
    # to histogram, as does xgboost when the package is missing or a class
    # is too small. Default 'xgboost'.
    'infection_intensity_strategy': 'xgboost',

    # (bool) - Apply log1p to features whose name contains 'intensity',
    # 'p75', 'p95' or 'max' (only when all their values are non-negative)
    # before standardising and embedding. Compresses the bright tail so a
    # handful of very bright cells stop dominating the embedding. Worth
    # enabling for wide-dynamic-range pathogen stains. Default False.
    'infection_pca_log_intensity': False,

    # (int) - Ceiling on cells fed to the embedding; above it a random
    # subsample of this size is drawn using a fixed seed of 0, independent
    # of infection_pca_random_state. Lower it when UMAP or t-SNE is slow
    # or memory-hungry, raise it to keep rare subpopulations. Applied
    # after non-finite rows are dropped. Default 50000.
    'infection_pca_max_cells': 50000,

    # (float) - Alert level for the ground-truth separation score - the
    # absolute difference, between the two clusters, in the fraction of
    # intensity-extreme cells that are infected (0-1). Dropping below it
    # only prints a warning; the cluster labels are still applied. Raise
    # it to be told sooner that the embedding is not separating infection.
    # Default 0.2.
    'infection_pca_min_gt_separation': 0.2,

    # (float) - Silhouette value below which the log prints a 'weak
    # cluster structure' warning with tuning hints. It does not reject or
    # re-run the clustering - the cluster-derived labels are applied
    # regardless - so treat it purely as an alert level. Silhouette runs
    # from -1 to 1. Default 0.05.
    'infection_pca_min_silhouette': 0.05,

    # (int) - Intended cluster count for the embedding-based infection
    # call. Not currently honoured: the pca/umap/tsne QC always runs
    # KMeans with exactly two clusters, one mapped to infected and one to
    # uninfected, so changing this has no effect on results. Default 2.
    'infection_pca_n_clusters': 2,

    # (float) - Multiplier applied to the standardised pathogen-channel
    # features before embedding. Above 1 it stretches the embedding along
    # pathogen intensity so KMeans splits infected from uninfected rather
    # than by morphology; 1.0 leaves all features weighted equally. Raise
    # it when the log reports weak cluster separation. Default 2.0.
    'infection_pca_pathogen_weight': 2.0,

    # (int) - Seed for KMeans and for the UMAP/t-SNE embeddings in the
    # pca/umap/tsne strategies. Fixing it makes the embedding and the
    # resulting infected/uninfected cluster assignment reproducible;
    # change it to check that the split is not an artifact of one
    # initialisation. Note the max-cells subsample uses its own fixed
    # seed. Default 42.
    'infection_pca_random_state': 42,

    # (list[float]) - Candidate t-SNE learning rates tried when
    # infection_pca_tsne_search is True. Too low leaves a dense ball with
    # points crowded together; too high scatters the map into a diffuse
    # cloud. Either way the infected/uninfected split blurs. Paired with
    # every perplexity candidate, so keep both lists short. Default
    # [200.0, 500.0].
    'infection_pca_tsne_learning_rate_grid': [200.0, 500.0],

    # (float) - Fixed t-SNE perplexity used when infection_pca_tsne_search
    # is False, automatically capped at max(5, (n_cells-1)/3). Lower
    # values emphasise local structure and can break one population into
    # several clumps; higher values emphasise global structure and merge
    # them. The learning rate is left at 'auto'. Default 30.0.
    'infection_pca_tsne_perplexity': 30.0,

    # (list[float]) - Candidate t-SNE perplexity values tried when
    # infection_pca_tsne_search is True - roughly how many neighbours each
    # point balances. Candidates at or above (n_cells-1)/3 are discarded,
    # and if none survive the code falls back to min(30, that cap). Small
    # values fragment clusters, large ones merge them. Default [15.0,
    # 30.0, 45.0].
    'infection_pca_tsne_perplexity_grid': [15.0, 30.0, 45.0],

    # (bool) - Fit t-SNE once per combination of
    # infection_pca_tsne_perplexity_grid and
    # infection_pca_tsne_learning_rate_grid, keeping the run that scores
    # highest on centroid distance times ground-truth separation. False
    # does a single fit at infection_pca_tsne_perplexity with
    # learning_rate 'auto'. Every extra grid point costs a full t-SNE fit.
    # Default True.
    'infection_pca_tsne_search': True,

    # (float) - Fixed UMAP min_dist used when infection_pca_umap_search is
    # False, between 0 and 1: the minimum spacing allowed between embedded
    # points. Near 0 gives tight, well-separated clumps that KMeans splits
    # cleanly; larger values spread points evenly and blur the
    # infected/uninfected boundary. Ignored during grid search. Default
    # 0.1.
    'infection_pca_umap_min_dist': 0.1,

    # (list[float]) - Candidate UMAP min_dist values tried when
    # infection_pca_umap_search is True, each between 0 and 1. Near 0
    # packs points tightly and gives crisper clusters for KMeans to split;
    # larger values spread points out and blur the boundary. Paired with
    # every n_neighbors candidate. Default [0.0, 0.05, 0.1, 0.3].
    'infection_pca_umap_min_dist_grid': [0.0, 0.05, 0.1, 0.3],

    # (int) - Fixed UMAP n_neighbors used when infection_pca_umap_search
    # is False - the size of the local neighbourhood UMAP tries to
    # preserve. Low values (5-10) favour local detail and can shatter one
    # population into several clumps; high values (30 and up) favour
    # global layout. Ignored during grid search. Default 15.
    'infection_pca_umap_n_neighbors': 15,

    # (list[int]) - Candidate UMAP n_neighbors values tried when
    # infection_pca_umap_search is True. Small values (around 5) preserve
    # local structure and split fine subpopulations; large values (30 and
    # up) emphasise global structure. Every entry is paired with every
    # value in infection_pca_umap_min_dist_grid, so keep the list short.
    # Default [5, 10, 15, 30].
    'infection_pca_umap_n_neighbors_grid': [5, 10, 15, 30],

    # (bool) - Fit UMAP once per combination of
    # infection_pca_umap_n_neighbors_grid and
    # infection_pca_umap_min_dist_grid, keeping the run with the highest
    # cluster-centroid distance times ground-truth separation. True costs
    # one UMAP fit per grid point; False does a single fit using
    # infection_pca_umap_n_neighbors and infection_pca_umap_min_dist.
    # Default True.
    'infection_pca_umap_search': True,

    # (float) - Upper edge of the discarded probability band, between 0
    # and 1. Together with infection_xgb_ambiguous_low it defines the
    # interval whose cells are dropped when infection_xgb_drop_ambiguous
    # is True. Lower it toward the threshold to keep more cells, raise it
    # to discard more. Swapped automatically if it falls below the low
    # bound. Default 0.75.
    'infection_xgb_ambiguous_high': 0.75,

    # (float) - Lower edge of the discarded probability band, between 0
    # and 1. Cells whose probability falls between this and
    # infection_xgb_ambiguous_high are dropped when
    # infection_xgb_drop_ambiguous is True. Raise it toward the threshold
    # to keep more cells, lower it to discard more borderline ones.
    # Swapped automatically if it exceeds the high bound. Default 0.25.
    'infection_xgb_ambiguous_low': 0.25,

    # (float) - Fraction of feature columns offered to each tree, between
    # 0 and 1. Lowering it stops a couple of dominant pathogen-intensity
    # features from being chosen by every tree, spreading gain across
    # morphology features and reducing overfitting; 1.0 exposes all
    # features to every tree. Default 0.8.
    'infection_xgb_colsample_bytree': 0.8,

    # (bool) - After prediction, discard cells whose probability lies
    # between infection_xgb_ambiguous_low and infection_xgb_ambiguous_high
    # instead of forcing a call on them. True gives cleaner infected vs
    # uninfected motility comparisons at the cost of sample size; False
    # keeps every cell. Only used by the xgboost strategy. Default True.
    'infection_xgb_drop_ambiguous': True,

    # (float) - Shrinkage applied to each boosting round's contribution
    # (XGBoost eta). Lower values need more rounds but give smoother,
    # better-calibrated infection probabilities; higher values converge
    # fast and can slam probabilities to 0 or 1, defeating the ambiguous
    # band. Usual range 0.01-0.3, tuned together with
    # infection_xgb_n_estimators. Default 0.1.
    'infection_xgb_learning_rate': 0.1,

    # (float) - Half-width of the confidence band around
    # infection_xgb_proba_threshold, clamped to 0-0.49. In 'relabel' mode
    # only cells outside the band get their label overridden, the rest
    # keep the mask-based call; in 'remove' mode cells inside the band are
    # spared deletion. Raise it to trust the model less. Default 0.15.
    'infection_xgb_margin': 0.15,

    # (int) - Maximum depth of each boosted tree. Deeper trees capture
    # interactions between morphology and pathogen-intensity features but
    # overfit the quartile-derived training labels; shallower trees
    # generalise better across wells. Typical range 2-8; raise it only
    # when the classifier cannot separate infected from uninfected.
    # Default 3.
    'infection_xgb_max_depth': 3,

    # (int) - Per well, how many intensity-extreme examples each class
    # must reach before that well's training data are balanced by
    # subsampling to the smaller class; wells that have both classes but
    # fewer examples contribute all of theirs, unbalanced. Wells with only
    # one class are skipped entirely. No well is ever excluded for being
    # small, so raising it leaves more wells unbalanced and the training
    # set more skewed - lower it towards 1 to force balancing in every
    # usable well. Default 10.
    'infection_xgb_min_cells_per_class': 10,

    # (int) - Number of boosting rounds (trees) trained, passed as
    # num_boost_round. More rounds fit the intensity-extreme training set
    # more tightly and push infection probabilities away from 0.5, which
    # shrinks the ambiguous band, but cost runtime and can overfit small
    # wells. Trade off against infection_xgb_learning_rate. Default 200.
    'infection_xgb_n_estimators': 200,

    # (int) - Threads XGBoost uses for training and prediction (its
    # nthread parameter). -1 uses every available core; set a small
    # positive number to leave CPU free for other work or when several
    # plates run at once. It changes runtime, not the training recipe.
    # Default -1.
    'infection_xgb_n_jobs': -1,

    # (str) - Column name the track-level ambiguous filter and the QC
    # probability plot look for. If that column is not in the table, BOTH
    # now fall back to discovering it -- the classifier writes
    # 'infection_prob', and until 2026-08-12 the fallback was unreachable,
    # so track-level ambiguous dropping silently never ran on a default
    # configuration. Set it explicitly only to override the discovery.
    # Default 'infection_xgb_proba'.
    'infection_xgb_proba_column': 'infection_xgb_proba',

    # (float) - Predicted probability at or above which a cell is called
    # infected, between 0 and 1. Lowering it makes infection calling more
    # permissive (more cells become infected), raising it more stringent.
    # It is also the centre of the confidence band whose half-width is
    # infection_xgb_margin. Default 0.5.
    'infection_xgb_proba_threshold': 0.5,

    # (int) - Seed for the generator that balances the per-well training
    # set, i.e. which intensity-extreme cells are sampled for each class.
    # It is not handed to XGBoost itself. Change it and re-run to confirm
    # the adjusted infection calls are stable under a different training
    # draw. Default 42.
    'infection_xgb_random_state': 42,

    # (float) - L2 penalty on leaf weights. Larger values shrink leaf
    # outputs, giving a more conservative model whose probabilities sit
    # closer to 0.5 and therefore more cells inside the ambiguous band; 0
    # removes the penalty entirely. Raise it when the model fits training
    # cells perfectly yet disagrees wildly with mask-based labels. Default
    # 1.0.
    'infection_xgb_reg_lambda': 1.0,

    # (float) - Fraction of training rows drawn at random for each
    # boosting round, between 0 and 1. Below 1 it injects stochasticity
    # that limits overfitting to the small set of intensity-extreme cells
    # used for training; 1.0 uses every training row every round. Lower it
    # if the classifier appears to memorise individual wells. Default 0.8.
    'infection_xgb_subsample': 0.8,

    # (int) - How many features, ranked by XGBoost gain, are retained for
    # the feature-importance panel of the QC figure. This is a display cut
    # applied after training: it never changes the model or the infection
    # calls. Lower it for a readable bar chart, raise it to inspect more
    # features. Default 20.
    'infection_xgb_top_features': 20,

    # (float) - Largest plausible centroid movement between consecutive
    # frames, in pixels. A single frame that jumps out and straight back
    # is interpolated from its neighbours; any other jump above this value
    # discards the whole track. Raise it for fast objects or sparse
    # timelapses, lower it to purge ID-swap artifacts. Default 50.0.
    'max_displacement': 50.0,

    # (bool) - Run the automated motility assay after segmentation: it
    # rebuilds per-object measurements from merged/*.npy, cleans tracks,
    # computes per-track velocity and straightness, applies the infection
    # QC, and writes motility_plots plus a well-level summary table. It
    # only fires when timelapse is also True, and it is what reveals the
    # Motility setting categories. Default False.
    'motility_analysis': False,

    # (tuple) - Spatial x-axis limits for the origin-centred track panels
    # (infected and uninfected) of the motility figure, in plotted
    # coordinate units - um when pixels_per_um is set, otherwise pixels -
    # not time. The whole-field all-tracks axis next to them always
    # autoscales from the data and ignores this setting. Set to None for
    # autoscaling. Default (100, -100), a 200-unit window written high-to-
    # low so the axis draws reversed.
    'motility_xlim': (100, -100),

    # (tuple) - Spatial y-axis limits for the origin-centred track panels
    # (infected and uninfected) of the motility figure, in plotted
    # coordinate units - um when pixels_per_um is set, otherwise pixels -
    # not velocity. The whole-field all-tracks axis next to them always
    # autoscales from the data and ignores this setting. Set to None for
    # autoscaling. Default (100, -100), a 200-unit window written high-to-
    # low so the axis draws reversed.
    'motility_ylim': (100, -100),

    # (int) - CPU workers for parallel stages: measurement, mask
    # adjustment, DataLoader loading, and the sklearn/UMAP calls where -1
    # means every core. Raise it to shorten CPU-bound steps until RAM or
    # disk I/O saturates. Note the measure-and-crop pipeline overrides
    # your value with cpu_count()-4. Defaults vary by pipeline:
    # cpu_count()-4, -1, or None.
    'n_jobs': 8,

    # (int or None) - Zero-indexed raw acquisition channel segmented into
    # nucleus masks, and the channel that nucleus_background,
    # nucleus_Signal_to_noise and remove_background_nucleus apply to. None
    # means no nucleus masks, hence no nucleus table, no cell-to-nucleus
    # linking, and nothing subtracted from the cytoplasm mask. Set it
    # whenever a DNA stain was acquired. Default None.
    'nucleus_channel': 0,

    # (int or None) - Zero-indexed raw acquisition channel segmented into
    # pathogen masks (Toxoplasma etc.), and the channel
    # pathogen_background, pathogen_Signal_to_noise and
    # remove_background_pathogen apply to. None disables pathogen
    # segmentation, the pathogen table, the infected-only filter
    # (uninfected) and the adjust_cells step, which needs cell, nucleus
    # and pathogen masks together. Default None.
    'pathogen_channel': 1,

    # (float) - Image scale in pixels per micrometre. Track coordinates
    # are divided by it, so plots switch from px to um, and together with
    # seconds_per_frame it converts velocity from px/frame to um/min. Take
    # it from the objective and camera pixel size rather than tuning it -
    # it rescales every reported velocity. Default 1.78.
    'pixels_per_um': 1.78,

    # (bool) - If measurements.db already holds the table named by
    # db_table_name, load it instead of re-extracting regionprops from
    # merged/*.npy. Saves most of the runtime when re-running only the
    # infection QC or the plots, but it also skips track smoothing, so
    # changes to max_displacement or zscore_thresh only take effect with
    # this set to False. Default True.
    'reuse_existing_measurements': True,

    # (int) - Interval between consecutive timelapse frames, in seconds.
    # Used with pixels_per_um to convert mean per-frame displacement into
    # um/min; if either is missing, velocities stay in px/frame. It is
    # also printed in the motility plot legend box. A wrong value rescales
    # every reported velocity linearly. Default 60.
    'seconds_per_frame': 60,

    # (str, path) - Folder the current step reads from and writes into:
    # raw images for mask generation, the merged/ folder of .npy stacks
    # for measure, the plate root for dataset/regression steps, or the
    # folder of .fastq.gz reads for sequencing. Outputs (stack/, masks/,
    # measurements/measurements.db, datasets/, results/) are created
    # inside it. A list of paths, or a "['a','b']" string, processes
    # several plates in one run. No usable default: the settings factories
    # fill a placeholder ('path' or '/path/to/src'), so this must be
    # supplied.
    'src': 'path',

    # (bool) - Actually apply the straightness cut. False only reports how
    # many tracks exceed straightness_threshold and changes nothing; True
    # removes those near-perfectly-straight tracks from the velocity
    # table, the per-well summary and the plots. Turn it on when stage
    # drift or identity swaps produce implausibly straight trajectories.
    # Default False.
    'straightness_filter': False,

    # (float) - Straightness cut-off, where straightness = net
    # displacement / total path length (0 = returns to start, 1 =
    # perfectly straight). When straightness_filter is True, tracks at or
    # above this value are dropped as drift or tracking artifacts, so
    # lowering it discards more tracks. The count is always logged.
    # Default 0.95.
    'straightness_threshold': 0.95,

    # (str) - Which object's feature block ({object}_* columns) the
    # XGBoost infection classifier trains on: 'cell', 'nucleus' or
    # 'pathogen'; anything else falls back to 'cell'. It does not change
    # what is tracked - track geometry and velocity always come from the
    # cell centroids. Default 'cell'.
    'tracked_object': 'cell',

    # (float) - Outlier sensitivity when smoothing scalar features within
    # a track (area, bbox area, equivalent diameter, perimeter, solidity,
    # mean/max/min intensity). A frame more than this many standard
    # deviations from its own track mean, whose two neighbours are both
    # within half that, is replaced by their average. Lower smooths more;
    # nothing is deleted. Default 3.0.
    'zscore_thresh': 3.0,
}

## 4. Run it

This is the long cell. Progress is logged; if you want more of it, raise the log levels in Preferences → Logging, or set `SPACR_LOG_LEVEL=DEBUG` before starting Jupyter.

In [ ]:
automated_motility_assay(settings)

## Where the output went

Per-track trajectories, speed and displacement summaries.

spaCR writes beside the source folder rather than into a global location, so a plate stays self-contained and re-running does not clobber a different experiment.

### Next steps

* The GUI covers the same workflows with the settings laid out as a form — `python -m spacr`.
* The narrated walkthroughs are at <https://einarolafsson.github.io/spacr/tutorials/>.
* The API reference is at <https://einarolafsson.github.io/spacr/>.